# 🧪 Analíticas de sangre (tabular) — v2 ÓPTIMO — ensemble de 4 modelos (CPU · modelo independiente)
## TFM · Módulo Tabular (Laboratorio) · Universidad de Salamanca

---

Versión que **mejora todo lo posible** dentro del techo real de la señal tabular (≈0.67-0.69; el EDA ya lo anticipaba).
Modelo **independiente** (sin fusión). TabPFN por **nube** (`tabpfn-client`).

## 🔄 Actualización — adaptación a las conclusiones del EDA

| Cambio | Detalle | Origen (recuadro naranja) |
|---|---|---|
| **Etiquetado FINAL** | `POS=(==1)` · `NEG=(==0)|(NaN→0)` · **−1 ENMASCARADO** (U-Ignore). **Se elimina** la derivación de negativos desde *No Finding* (sesgo de espectro). | §3 Definición del negativo |
| **Métrica primaria = AUC-PR** | `macro_AP_path` gobierna tuning, early-stopping y selección; **AUC-ROC pasa a secundaria**; **IC bootstrap** en test. | §3 Métrica · §1 IC por val/test pequeños |
| **Escalado robusto** | `RobustScaler` (mediana/IQR) en la rama densa: la edad **no es gaussiana** (Shapiro p<0,001). | §2 Demografía |
| **Raw vs percentil** | Verificado: **árboles = raw** (NaN nativo), **densos = percentil**. Nunca ambos (r>0,90). | §9 Redundancia raw–percentil |
| **NaN amplios** | En la rama densa, las analíticas con >55 % ausente entran **solo por su flag**; las prioritarias se protegen. | §4 Ausencias |
| **Clusters fisiológicos** | Scores de severidad por cluster (renal, eritrocitario, ácido-base…) como features; reducción por representante bajo toggle `USE_CLUSTER_REPR` (A/B, OFF por defecto). | §8 Estructura multivariante |
| **Equidad** | Métricas estratificadas por sexo, etnia e ingreso exportadas a CSV. | §2/§9 Equidad |

Se mantiene: **sin data leakage** (imputación/escalado por fold), **sin `cxr_view`**, flags de missingness (MNAR),
`scale_pos_weight`/`pos_weight` por etiqueta, calibración isotónica y OOF sin fuga.

## ⏱️ Coste (CPU): ~1 hora (tuning de los 4 modelos + K=5 + TabPFN nube).

## 🎯 Expectativa honesta

La señal tabular tiene techo (~0.67-0.69 AUC). Su valor real es como **modalidad complementaria** en la fusión, no en
solitario (§5/§8 del EDA: perfiles solapados, t-SNE sin clusters separables).
**Aviso:** al cambiar la definición del negativo y la métrica, los números **no son comparables** con los de la
ejecución anterior; hay que reentrenar y volver a documentar.

In [ ]:
# CELDA 1 · DEPENDENCIAS
import subprocess, sys
try:
    import torch  # noqa: F401
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","torch","--index-url","https://download.pytorch.org/whl/cpu","-q"], check=True)
for pkg in ["xgboost","lightgbm","scikit-learn","optuna","pandas","numpy","matplotlib","seaborn","tqdm"]:
    subprocess.run([sys.executable,"-m","pip","install",pkg,"-q"], check=True)
try:
    subprocess.run([sys.executable,"-m","pip","install","tabpfn-client","-q"], check=True); print("tabpfn-client (nube) instalado.")
except Exception as ex:
    print("tabpfn-client no se pudo instalar (se omitirá):", ex)
print("Dependencias instaladas.")


In [ ]:
# CELDA 2 · IMPORTS / SEMILLAS / DISPOSITIVO
import os, gc, json, time, copy, random, warnings, functools
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             confusion_matrix, roc_curve, precision_recall_curve)
import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING); warnings.filterwarnings("ignore")

# ── TabPFN (modelo en la NUBE) · activación condicionada al TOKEN ──────────────────────────────
# IMPORTANTE: TabPFN solo se activa si existe la variable de entorno TABPFN_TOKEN.
# Motivo: `tabpfn_client` se importa correctamente aunque NO haya credenciales, y al predecir
# lanza un login INTERACTIVO por stdin. En ejecución por lotes (jupyter nbconvert) no hay stdin,
# el kernel se queda bloqueado y MUERE ("DeadKernelError: Kernel died") sin traceback de Python.
# Condicionar la activación al token hace el notebook reproducible en batch.
# Para usar TabPFN:  set TABPFN_TOKEN=<tu_token>   antes de lanzar el notebook.
TABPFN_OK = False
_tok = os.environ.get("TABPFN_TOKEN")
if _tok:
    try:
        import tabpfn_client
        from tabpfn_client import TabPFNClassifier
        tabpfn_client.set_access_token(_tok)
        TABPFN_OK = True
    except Exception as _e:
        print("tabpfn-client no disponible:", _e); TABPFN_OK = False
else:
    print("TABPFN_TOKEN no definido -> TabPFN DESACTIVADO (el resto del ensemble no se ve afectado).")

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(4); DEVICE=torch.device("cpu")
print(f"xgboost {xgb.__version__} · lightgbm {lgb.__version__} · TabPFN: {TABPFN_OK}")

In [ ]:
# CELDA 3 · RUTAS, ETIQUETAS Y CONSTANTES (v2 · etiquetado FINAL)
BASE=Path(r"C:\TFM\1.Opción - Symile Mimic\symile-mimic-a-multimodal-clinical-dataset-of-chest-x-rays-electrocardiograms-and-blood-labs-from-mimic-iv-1.0.0")
CSV_DIR=BASE/"data_csv"/"clean"
TRAIN_CSV,VAL_CSV,TEST_CSV=CSV_DIR/"train_clean.csv",CSV_DIR/"val_clean.csv",CSV_DIR/"test_clean.csv"
OUTPUT_DIR=Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\03_labs\v2"); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
FIG_DIR=OUTPUT_DIR/"figuras"; FIG_DIR.mkdir(parents=True,exist_ok=True)
print("Salidas en:", OUTPUT_DIR)
LABELS=["Atelectasis","Cardiomegaly","Edema","Lung Opacity","No Finding","Pleural Effusion"]
N_LABELS=len(LABELS); NO_FINDING="No Finding"
PATHOLOGY_LABELS=[l for l in LABELS if l!=NO_FINDING]; CORE_LABELS=["Cardiomegaly","Edema","Pleural Effusion"]

# ── ETIQUETADO FINAL (§3 EDA) ────────────────────────────────────────────────
# POS=(==1) · NEG=(==0)|(NaN→0) · −1 ENMASCARADO (U-Ignore) · «No Finding» NO como negativo.
DERIVE_NEGATIVES_FROM_EXCLUSIVITY=False   # ← antes True; la derivación introduce SESGO DE ESPECTRO
FLAG_THRESH=0.02                          # umbral para crear flag de missingness (MNAR, §6 EDA)
WIDE_NAN_THRESH=0.55                      # >55 % ausente → en modelos DENSOS solo se usa el flag (§4 EDA)
USE_CLUSTER_REPR=False                    # toggle A/B de reducción por clusters fisiológicos (§8 EDA)
K_COMPARE=5
TUNE_GBM=30; TUNE_LOGREG=20; TUNE_MLP=20
USE_TABPFN=TABPFN_OK; TABPFN_MAX=4000; TABPFN_MAX_FINAL=6000
GENDER_MAP={0:0,1:1,"0":0,"1":1,"M":1,"F":0}
RACE_MAP={"UNKNOWN":0,"WHITE":1,"BLACK":2,"ASIAN":3,"HISPANIC_LATINO":4,"OTHER_KNOWN":0}
ADMISSION_MAP={"SCHEDULED":0,"EMERGENCY":1,"OBSERVATION":2,"URGENT":3}
ADM_LOC_MAP={"EMERGENCY_ROOM":0,"REFERRAL":1,"TRANSFER":2,"INTRA_HOSPITAL":3}
RACE_ONEHOT=5
LAB_MEANING={
 "urea_nitrogen":"función renal/gravedad","albumin":"proteína; baja en efusiones/ICC","rdw":"anisocitosis; fallo cardíaco",
 "creatinine":"función renal","lymphocytes_pct":"inflamación/inmunidad","hemoglobin":"anemia","hematocrit":"anemia",
 "rbc":"masa eritrocitaria","mch":"índice eritrocitario","mchc":"índice eritrocitario","mcv":"índice eritrocitario",
 "nrbc_abs":"estrés medular","pt":"coagulación","inr_pt":"coagulación/hepático","ptt":"coagulación",
 "phosphate":"metabólico","calcium_total":"metabólico","magnesium":"metabólico","sodium":"electrolito/renal",
 "potassium":"electrolito/renal","chloride":"electrolito","bicarbonate":"ácido-base","anion_gap":"ácido-base",
 "base_excess":"ácido-base","ph":"ácido-base","pco2":"gases","po2":"gases","calculated_total_co2":"ácido-base",
 "lactate":"hipoperfusión","glucose":"metabólico","alt":"hepático","ast":"hepático",
 "alkaline_phosphatase":"hepático/óseo","bilirubin_total":"hepático","fibrinogen":"coagulación/inflamación",
 "wbc_count":"inflamación","neutrophils_pct":"inflamación","monocytes_pct":"inmunidad",
 "basophils_pct":"inmunidad","eosinophils_pct":"alergia/inmunidad","platelet_count":"hemostasia"}

# ── CLUSTERS FISIOLÓGICOS (§8 EDA · dendrograma de Ward) ──────────────────────
# ORIGEN EDA: recuadro naranja "existen clusters fisiológicos (eritrocitario, renal, ácido-base,
#             leucocitario) → posible reducción por representante para mejorar la generalización".
# Se agrupan por el significado clínico ya declarado en LAB_MEANING (no a mano).
LAB_CLUSTERS={
 "eritrocitario":["hemoglobin","hematocrit","rbc","mch","mchc","mcv","rdw","nrbc_abs"],
 "renal":["urea_nitrogen","creatinine","sodium","potassium","chloride"],
 "acido_base":["bicarbonate","anion_gap","base_excess","ph","calculated_total_co2","pco2","po2","lactate"],
 "leucocitario":["wbc_count","neutrophils_pct","lymphocytes_pct","monocytes_pct","basophils_pct","eosinophils_pct"],
 "coagulacion":["pt","inr_pt","ptt","fibrinogen","platelet_count"],
 "hepatico":["alt","ast","alkaline_phosphatase","bilirubin_total","albumin"],
 "metabolico":["glucose","phosphate","calcium_total","magnesium"]}
# PRIORITARIAS (§5/§10 EDA): nunca se descartan por la regla de NaN amplio.
PRIORITY_LABS=["urea_nitrogen","albumin","rdw","creatinine","lymphocytes_pct"]

def readable(col):
    name,code=col.rsplit("_",1); pretty=name.replace("_"," ").title(); mn=LAB_MEANING.get(name,"")
    return f"{pretty} ({code})"+(f" – {mn}" if mn else "")
print(f"Constantes listas (etiquetado FINAL · derive={DERIVE_NEGATIVES_FROM_EXCLUSIVITY} · clusters={len(LAB_CLUSTERS)} · USE_CLUSTER_REPR={USE_CLUSTER_REPR}).")

In [ ]:
# CELDA 4 · CARGA DE CSV
df_train=pd.read_csv(TRAIN_CSV,sep=";"); df_val=pd.read_csv(VAL_CSV,sep=";"); df_test=pd.read_csv(TEST_CSV,sep=";")
DEMO=["subject_id","hadm_id","cxr_path","ecg_path","age","gender","race","admission_type","admission_location","cxr_view","hours_adm_to_cxr"]
RAW_LABS=[c for c in df_train.columns if c not in LABELS and c not in DEMO and "pctile" not in c]
PCT_LABS=[c for c in df_train.columns if "pctile" in c]
print(f"train={len(df_train)} val={len(df_val)} test={len(df_test)} | raw={len(RAW_LABS)} percentil={len(PCT_LABS)}")


In [ ]:
# CELDA 5 · OBJETIVOS — definición FINAL del negativo (U-IGNORE del −1)
# ══ build_targets ════════════════════════════════════════════════════════════
# QUÉ HACE: convierte los estados 1/0/−1/NaN de cada etiqueta en (labels, mask).
# FINALIDAD: fijar la definición FINAL acordada con el tutor:
#            POS=(==1) · NEG=(==0)|(NaN→0) · −1 ENMASCARADO (U-Ignore de CheXpert).
# ORIGEN EDA: §3 · recuadro naranja "negativo = 0 explícito + NaN→0; el −1 se enmascara;
#            «Sin hallazgo» NO como negativo (evita el sesgo de espectro)".
# CAMBIO vs versión previa: se ELIMINA la derivación de negativos desde «No Finding».
#            El NaN pasa a negativo (mask=1) en lugar de quedar enmascarado.
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc post-resultados): comparar AUC-PR
#            con/sin esta definición; verificar pos_weight ≈ 2,28/1,85/3,43/2,15/6,31/1,66.
#            Unifica el negativo con CXR y ECG → la fusión combina probabilidades coherentes.
def build_targets(df,uncertainty_policy="ignore",derive=DERIVE_NEGATIVES_FROM_EXCLUSIVITY,verbose=False):
    raw=df[LABELS].to_numpy(dtype=float); N=raw.shape[0]
    labels=(raw==1.0).astype(np.float32)                 # 1 → positivo ; 0 y NaN → 0 (negativo)
    mask  =np.ones((N,N_LABELS),np.float32)              # por defecto TODO entra en pérdida/métrica
    unc=(raw==-1.0)
    if   uncertainty_policy=="ignore": mask[unc]=0.0      # −1 → ENMASCARADO (definición FINAL)
    elif uncertainty_policy=="ones":   labels[unc]=1.0    # −1 → positivo (alternativa, no usada)
    # "zeros": −1 → negativo (ya es 0 en labels)
    n_dp=n_dn=0
    if derive:  # CONSERVADO por compatibilidad; NO forma parte de la definición FINAL
        nf=LABELS.index(NO_FINDING); pc=[j for j in range(N_LABELS) if j!=nf]; nfp=(raw[:,nf]==1)
        for j in pc:
            f=nfp&np.isnan(raw[:,j]); labels[f,j]=0.0; n_dp+=int(f.sum())
        ap=(raw[:,pc]==1).any(1); fn=ap&np.isnan(raw[:,nf]); labels[fn,nf]=0.0; n_dn=int(fn.sum())
    if verbose:
        obs=mask==1
        print(f"   pos={int((labels*mask).sum()):,} · neg={int(((labels==0)&obs).sum()):,} · enmascarados(−1)={int((mask==0).sum()):,}"
              + (f" · derivados(off)={n_dp+n_dn:,}" if derive else ""))
        for j,l in enumerate(LABELS):
            s=mask[:,j]==1; p=int((labels[s,j]==1).sum()); n=int((labels[s,j]==0).sum())
            print(f"      {l:18s} pos={p:5d} neg={n:5d} pos_weight={n/max(p,1):.2f}")
    return labels,mask
y_train,m_train=build_targets(df_train,"ignore",verbose=True)
y_val,m_val=build_targets(df_val,"ignore"); y_test,m_test=build_targets(df_test,"ignore")

In [ ]:
# CELDA 6 · FEATURES + INGENIERÍA CLÍNICA (sin leakage) — decisiones derivadas del EDA
# RESUMEN DE DECISIONES (todas con origen en un recuadro naranja del EDA):
#  · RAW vs PERCENTIL (§9): raw≡percentil (r>0,90) → NUNCA los dos a la vez. ÁRBOLES=raw
#    (invariantes a escala + NaN nativo); DENSOS=percentil (ya normalizado 0–1). Se VERIFICA abajo.
#  · NaN AMPLIOS (§4): ninguna supera el 70 %. En DENSOS, las de >WIDE_NAN_THRESH se sustituyen por
#    su flag (el valor es demasiado disperso; su ausencia sí informa). Las PRIORITARIAS se protegen.
#  · MNAR (§6): flags de missingness para las de >FLAG_THRESH ausente.
#  · CLUSTERS (§8): scores de severidad por cluster fisiológico; reducción por representante bajo
#    toggle USE_CLUSTER_REPR (A/B), apagada por defecto para no degradar el resultado actual.
#  · ESCALADO (§2): la edad NO es gaussiana (Shapiro p<0,001) → RobustScaler, no StandardScaler.
from sklearn.preprocessing import RobustScaler
train_miss=df_train[RAW_LABS].isna().mean()
FLAG_LABS=[c for c in RAW_LABS if train_miss[c]>FLAG_THRESH]
_base=lambda c: c.rsplit("_",1)[0]
_item=lambda c: c.rsplit("_",1)[1]
# Verificación C4: cada analítica aporta UNA representación por rama (raw en árbol, pct en denso).
PCT_OF={_item(c):[p for p in PCT_LABS if p.endswith(_item(c))] for c in RAW_LABS}
_dup=[c for c in RAW_LABS if len(PCT_OF[_item(c)])>1]
print(f"[C4] raw={len(RAW_LABS)} · percentil={len(PCT_LABS)} · analíticas con >1 percentil: {len(_dup)} (esperado 0)")
# NaN amplios: solo afecta a la rama DENSA y nunca a las prioritarias
WIDE_NAN_LABS=[c for c in RAW_LABS if train_miss[c]>WIDE_NAN_THRESH and _base(c) not in PRIORITY_LABS]
DENSE_PCT_KEEP=[p for p in PCT_LABS if not any(p.endswith(_item(c)) for c in WIDE_NAN_LABS)]
print(f"[NaN amplio >{WIDE_NAN_THRESH:.0%}] {len(WIDE_NAN_LABS)} analíticas → en DENSO solo su flag; percentiles usados: {len(DENSE_PCT_KEEP)}/{len(PCT_LABS)}")

def _col(key):
    for c in RAW_LABS:
        if c.startswith(key+"_") or c==key: return c
    return None
C_UREA=_col("urea_nitrogen"); C_CREAT=_col("creatinine"); C_NEUT=_col("neutrophils_pct"); C_LYMPH=_col("lymphocytes_pct"); C_RDW=_col("rdw")

# ══ clinical_ratios ══════════════════════════════════════════════════════════
# QUÉ HACE: construye ratios clínicos a partir del valor bruto.
# FINALIDAD: capturar señal fisiológica que una analítica aislada no da: BUN/Creatinina (función
#            renal), Neutrófilos/Linfocitos (inflamación), RDW×Edad (riesgo cardio).
# ORIGEN EDA: §5/§10 · recuadro naranja "priorizar Urea, Albúmina, RDW, Creatinina y añadir ratios
#            clínicos (BUN/Creatinina, RDW×Edad) por su fundamento fisiológico".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): ¿superan los ratios a sus componentes en la
#            importancia? Si RDW×Edad destaca, reforzar el eje cardio-renal en la discusión.
def clinical_ratios(df):
    out=[]; names=[]
    def g(c): return df[c].to_numpy(np.float32) if c else np.full(len(df),np.nan,np.float32)
    urea,creat,neut,lymph,rdw=g(C_UREA),g(C_CREAT),g(C_NEUT),g(C_LYMPH),g(C_RDW)
    age=df["age"].astype(float).to_numpy(np.float32)
    out.append((urea/(creat+1e-6))[:,None]); names.append("BUN/Creatinina (ratio renal)")
    out.append((neut/(lymph+1e-6))[:,None]); names.append("Neutrófilos/Linfocitos (ratio inflam.)")
    out.append((rdw*age/100.0)[:,None]); names.append("RDW×Edad (riesgo cardio)")
    return np.hstack(out).astype(np.float32), names

# ══ cluster_scores ═══════════════════════════════════════════════════════════
# QUÉ HACE: resume cada cluster fisiológico en un único score = media de los percentiles del grupo.
# FINALIDAD: reducir la redundancia intra-cluster (colinealidad) con una variable interpretable
#            por eje clínico (renal, eritrocitario, ácido-base, leucocitario...).
# ORIGEN EDA: §8 · recuadro naranja "clusters fisiológicos → posible reducción por representante
#            para mejorar la generalización" (+ §4, bloques de correlación).
# NOTA: se AÑADEN como features extra; la SUSTITUCIÓN de las originales solo ocurre si
#       USE_CLUSTER_REPR=True (comparación A/B). Por defecto OFF para no degradar el resultado.
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): si un score de cluster aparece alto en importancia,
#       es evidencia de que el eje fisiológico completo (no una analítica suelta) es lo que discrimina.
def cluster_scores(df):
    cols=[]; names=[]
    for cname,members in LAB_CLUSTERS.items():
        pcs=[p for m in members for c in RAW_LABS if _base(c)==m for p in PCT_OF[_item(c)]]
        if not pcs: continue
        cols.append(np.nanmean(df[pcs].to_numpy(np.float32),axis=1)[:,None]); names.append(f"score:{cname}")
    if not cols: return np.zeros((len(df),0),np.float32),[]
    return np.nan_to_num(np.hstack(cols),nan=0.5).astype(np.float32), names
_CS_tr,CLUSTER_NAMES=cluster_scores(df_train)
print(f"[Clusters] scores de severidad añadidos: {CLUSTER_NAMES}")
RATIO_tr,RATIO_NAMES=clinical_ratios(df_train)
print(f"Ratios clínicos: {RATIO_NAMES}")
print(f"Flags de missingness: {len(FLAG_LABS)}/{len(RAW_LABS)}")

# ══ build_demo ═══════════════════════════════════════════════════════════════
# QUÉ HACE: matriz demográfica (edad, sexo, raza/ingreso/lugar one-hot, horas ingreso→CXR).
# FINALIDAD: incorporar la EDAD, uno de los dos predictores tabulares más fuertes.
# ORIGEN EDA: §2 · "«Desconocida» es categoría registrada, no NaN → one-hot, no imputar";
#            §10 · el ranking IVF sitúa Linfocitos % (0,535) y Edad (0,534) EMPATADOS en cabeza.
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): evaluar equidad por sexo/etnia (§9, V de Cramér
#            ≤0,05: diferencias significativas pero de tamaño despreciable).
def build_demo(df):
    N=len(df); cols=[]; names=[]
    cols.append(df["age"].astype(float).to_numpy()[:,None]); names.append("Edad")
    cols.append(df["gender"].map(lambda v:float(GENDER_MAP.get(v,0))).to_numpy()[:,None]); names.append("Sexo")
    def onehot(series,mp,pref):
        M=np.zeros((N,max(mp.values())+1),np.float32)
        for i,v in enumerate(series): M[i,mp.get(str(v).upper(),0)]=1.0
        return M,[f"{pref}={k}" for k,_ in sorted(mp.items(),key=lambda x:x[1])][:M.shape[1]]
    for col,mp,pref in [("race",RACE_MAP,"Raza"),("admission_type",ADMISSION_MAP,"Ingreso"),("admission_location",ADM_LOC_MAP,"Lugar")]:
        M,nm=onehot(df[col],mp,pref); cols.append(M); names+=nm
    cols.append(df["hours_adm_to_cxr"].astype(float).to_numpy()[:,None]); names.append("Horas ingreso→CXR")
    return np.hstack(cols).astype(np.float32), names
DEMO_M_tr,DEMO_NAMES=build_demo(df_train)

# ══ flags ════════════════════════════════════════════════════════════════════
# QUÉ HACE: indicadores binarios de "esta analítica falta".
# FINALIDAD: separar explícitamente la AUSENCIA del VALOR y capturar la señal MNAR.
# ORIGEN EDA: §6 · recuadro naranja "los positivos tienen menos ausencias → incluir flags".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): vigilar que el modelo no se apoye en exceso en los
#            flags (atajo espurio "tiene analíticas ⇒ está enfermo"); revisar su importancia relativa.
def flags(df): return df[FLAG_LABS].isna().astype(np.float32).to_numpy()

# ══ build_tree / build_dense ═════════════════════════════════════════════════
# QUÉ HACE: ensamblan la matriz de entrada de cada familia de modelos.
# FINALIDAD: dar a cada familia la representación que mejor le encaja (§9 EDA):
#            ÁRBOLES → RAW (NaN nativo, escala clínica) ; DENSOS → PERCENTIL (0–1, sin outliers).
# ORIGEN EDA: §9 "raw ≡ percentil (r>0,90) → usar una sola representación" · §4 "los árboles manejan
#            el NaN nativamente; para modelos densos, imputar por fold (sin fuga)".
def build_tree(df,demo):
    # Árboles: RAW completo (incl. NaN amplios: el árbol usa el NaN como información) + ratios + flags + demo
    parts=[df[RAW_LABS].to_numpy(np.float32), clinical_ratios(df)[0], flags(df), demo]
    if USE_CLUSTER_REPR: parts.insert(1, cluster_scores(df)[0])
    return np.hstack(parts)
def build_dense_raw(df,demo):
    # Densos: PERCENTIL (excluye NaN amplios, que entran solo vía flag) + scores de cluster + ratios + flags + demo
    # A/B MEDIDO (2026-07-21, salidas/03_labs/ab_cluster_scores.json): los 7 scores de cluster
    # NO aportan. LogReg OvR 3-fold sobre la matriz densa: CON=0,4152 vs SIN=0,4148 de macro AUC-PR;
    # diferencia +0,0005 con IC95% [-0,0010, +0,0019], que INCLUYE EL 0. Criterio fijado antes de
    # medir: si el IC incluye 0 -> quitarlos por parsimonia. Por eso ya NO entran por defecto.
    # Se conservan bajo USE_CLUSTER_REPR para poder reproducir el experimento.
    parts=[df[DENSE_PCT_KEEP].to_numpy(np.float32), clinical_ratios(df)[0], flags(df), demo]
    if USE_CLUSTER_REPR: parts.insert(1, cluster_scores(df)[0])
    return np.hstack(parts)
TREE_tr=build_tree(df_train,DEMO_M_tr); TREE_te=build_tree(df_test,build_demo(df_test)[0]); TREE_vl=build_tree(df_val,build_demo(df_val)[0])
DENSE_tr=build_dense_raw(df_train,DEMO_M_tr); DENSE_te=build_dense_raw(df_test,build_demo(df_test)[0]); DENSE_vl=build_dense_raw(df_val,build_demo(df_val)[0])
TREE_NAMES=([readable(c) for c in RAW_LABS]+(CLUSTER_NAMES if USE_CLUSTER_REPR else [])
            +RATIO_NAMES+[f"falta:{_base(c).replace('_',' ').title()}" for c in FLAG_LABS]+DEMO_NAMES)

# ══ dense_fit / dense_tx ═════════════════════════════════════════════════════
# QUÉ HACE: ajusta imputación (mediana) + escalado ROBUSTO usando SOLO el fold de entrenamiento.
# FINALIDAD: evitar data leakage y no asumir normalidad en variables asimétricas.
# ORIGEN EDA: §2 · recuadro naranja "la edad no es gaussiana (Shapiro-Wilk p<0,001, asimetría −0,58):
#            no usar modelos/escalados que asuman normalidad" → RobustScaler (mediana/IQR) en lugar
#            de StandardScaler. §6 · "missingness consistente entre splits (R≈0,99) → la imputación
#            aprendida en train se aplica a val/test sin reajuste ni fuga".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): comparar el rendimiento del MLP/LogReg con
#            RobustScaler frente al StandardScaler previo (cambio barato, efecto medible).
def dense_fit(idx):
    imp=SimpleImputer(strategy="median").fit(DENSE_tr[idx])
    sc=RobustScaler().fit(imp.transform(DENSE_tr[idx]))   # ← antes StandardScaler (edad no gaussiana)
    return imp,sc
def dense_tx(imp,sc,X): return sc.transform(imp.transform(X)).astype(np.float32)
print(f"TREE={TREE_tr.shape} · DENSE={DENSE_tr.shape} · escalado ROBUSTO + imputación POR FOLD (sin fuga)")


In [ ]:
# CELDA 7 · MÉTRICAS — PRIMARIA = AUC-PR (Average Precision); ROC secundaria
# ══ multilabel_metrics ═══════════════════════════════════════════════════════
# QUÉ HACE: por etiqueta calcula AUC-PR (AP), AUC-ROC, F1, sens/spec y confusión, respetando la
#           máscara (los −1 no entran).
# FINALIDAD: seleccionar modelos y hacer early-stopping con la métrica correcta para datos
#            desbalanceados: la AP penaliza los falsos positivos donde el ROC es demasiado optimista.
# ORIGEN EDA: §3 · recuadro naranja "métrica primaria AUC-PR por patología + IC bootstrap;
#            secundaria AUC-ROC; NO comparar AP entre patologías de distinta prevalencia".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): la línea base de la AP es la PREVALENCIA de cada
#            etiqueta (§3), así que una AP de 0,34 en «Sin hallazgo» (prev. 13,7 %) puede ser mejor
#            que una de 0,73 en Derrame (prev. 36 %). Reportar siempre AP junto a su prevalencia.
def multilabel_metrics(probs,labels,mask,thresholds=None):
    if thresholds is None: thresholds={l:0.5 for l in LABELS}
    res={}
    for j,l in enumerate(LABELS):
        s=mask[:,j]==1; yt=labels[s,j]; yp=probs[s,j]; npos=int(yt.sum()); nneg=int((1-yt).sum())
        thr=thresholds.get(l,0.5); pred=(yp>=thr).astype(float)
        if npos>=2 and nneg>=2: auc=roc_auc_score(yt,yp); ap=average_precision_score(yt,yp)
        else: auc=ap=float("nan")
        tp=int(((pred==1)&(yt==1)).sum()); tn=int(((pred==0)&(yt==0)).sum())
        fp=int(((pred==1)&(yt==0)).sum()); fn=int(((pred==0)&(yt==1)).sum())
        res[l]={"AUC":auc,"AP":ap,"F1":f1_score(yt,pred,zero_division=0),"sens":tp/max(tp+fn,1),"spec":tn/max(tn+fp,1),
                "n_pos":npos,"n_neg":nneg,"thr":thr,"TP":tp,"TN":tn,"FP":fp,"FN":fn,
                "prevalencia":npos/max(npos+nneg,1)}
    def mac(g,k):
        v=[res[l][k] for l in g if not np.isnan(res[l][k])]; return float(np.mean(v)) if v else float("nan")
    res["macro_AUC_core"]=mac(CORE_LABELS,"AUC"); res["macro_AUC_path"]=mac(PATHOLOGY_LABELS,"AUC")  # secundaria
    res["macro_AP_core"] =mac(CORE_LABELS,"AP");  res["macro_AP_path"] =mac(PATHOLOGY_LABELS,"AP")   # PRIMARIA
    return res

# ══ bootstrap_ap_ci ══════════════════════════════════════════════════════════
# QUÉ HACE: IC percentil del AUC-PR por remuestreo bootstrap sobre el conjunto evaluado.
# FINALIDAD: acompañar SIEMPRE la AP de su incertidumbre.
# ORIGEN EDA: §1 · recuadro naranja "val (750) y test (464) son pequeños en términos absolutos, lo
#            que introduce varianza en las métricas y exige IC por bootstrap".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): si los IC de dos modelos se solapan, NO se puede
#            afirmar que uno sea mejor; esto condiciona la comparación mono-modelo vs fusión.
def bootstrap_ap_ci(probs,labels,mask,n_boot=1000,alpha=0.05,seed=SEED):
    rng=np.random.RandomState(seed); out={}
    for j,l in enumerate(LABELS):
        s=np.where(mask[:,j]==1)[0]; yt=labels[s,j]; yp=probs[s,j]
        if int(yt.sum())<2 or int((1-yt).sum())<2: out[l]=(float("nan"),float("nan")); continue
        vals=[]
        for _ in range(n_boot):
            idx=rng.randint(0,len(s),len(s))
            if yt[idx].sum()<1 or (1-yt[idx]).sum()<1: continue
            vals.append(average_precision_score(yt[idx],yp[idx]))
        out[l]=(float(np.percentile(vals,100*alpha/2)),float(np.percentile(vals,100*(1-alpha/2)))) if vals else (float("nan"),float("nan"))
    return out

# ══ best_thresholds_by_f1 ════════════════════════════════════════════════════
# QUÉ HACE: umbral por etiqueta que maximiza F1 sobre el conjunto dado (se usa VAL).
# FINALIDAD: fijar puntos de operación en VALIDACIÓN, nunca en test.
# ORIGEN EDA: §11 · "TRAIN entrena; VAL calibra+umbrales+selección; TEST una sola vez".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): además del umbral F1, el EDA pide reportar un punto
#            de ALTA SENSIBILIDAD (cribado) y otro de ALTA ESPECIFICIDAD (confirmación).
def best_thresholds_by_f1(probs,labels,mask,grid=None):
    if grid is None: grid=np.linspace(0.05,0.95,37)
    thr={}
    for j,l in enumerate(LABELS):
        s=mask[:,j]==1; yt=labels[s,j]; yp=probs[s,j]
        if yt.sum()<2: thr[l]=0.5; continue
        bf,bt=-1,0.5
        for t in grid:
            f=f1_score(yt,(yp>=t).astype(float),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[l]=float(bt)
    return thr
print("Métricas listas (PRIMARIA = macro_AP_path · IC bootstrap disponible).")

In [ ]:
# CELDA 7b · KIT DE EVALUACIÓN CLÍNICA (B1–B8) — implementa los recuadros naranjas del EDA que faltaban
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.feature_selection import mutual_info_classif, f_classif

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · bootstrap_ci_metric                                          [B4]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : remuestrea con reemplazo y recalcula la métrica, devolviendo el intervalo percentil,
#              tanto POR ETIQUETA como para el MACRO de las 5 patologías.
# POR QUÉ    : con 464 pacientes en test una diferencia entre modelos puede ser puro azar; el IC es
#              lo que permite afirmar (o no) que un modelo supera a otro.
# ENTRADAS   : probs (N,6) · labels (N,6) · mask (N,6) · metric "ap"|"auc" · n_boot · alpha
# SALIDAS    : dict {etiqueta:(lo,hi)} + clave "macro_path" con el IC del macro
# ORIGEN EDA : §1 · recuadro naranja "reportar SIEMPRE IC bootstrap por el tamaño reducido de val/test".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              si los IC de dos arquitecturas (o de fusión vs mejor mono) SE SOLAPAN, decir
#              explícitamente que la mejora NO es concluyente.
# ══════════════════════════════════════════════════════════════════════════════
def bootstrap_ci_metric(probs, labels, mask, metric="ap", n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.RandomState(seed)
    scorer = average_precision_score if metric == "ap" else roc_auc_score
    out, macro_vals = {}, []
    for j, l in enumerate(LABELS):
        idx = np.where(mask[:, j] == 1)[0]; yt, yp = labels[idx, j], probs[idx, j]
        if int(yt.sum()) < 2 or int((1 - yt).sum()) < 2: out[l] = (float("nan"),)*2; continue
        vals = []
        for _ in range(n_boot):
            bs = rng.randint(0, len(idx), len(idx))
            if yt[bs].sum() < 1 or (1 - yt[bs]).sum() < 1: continue
            vals.append(scorer(yt[bs], yp[bs]))
        out[l] = (float(np.percentile(vals, 100*alpha/2)), float(np.percentile(vals, 100*(1-alpha/2)))) if vals else (float("nan"),)*2
    for _ in range(n_boot):   # el macro se remuestrea por PACIENTE (respeta la correlación entre salidas)
        bs = rng.randint(0, len(probs), len(probs)); per = []
        for j, l in enumerate(LABELS):
            if l not in PATHOLOGY_LABELS: continue
            sel = mask[bs, j] == 1; yt, yp = labels[bs][sel, j], probs[bs][sel, j]
            if yt.sum() < 1 or (1 - yt).sum() < 1: continue
            per.append(scorer(yt, yp))
        if per: macro_vals.append(np.mean(per))
    out["macro_path"] = (float(np.percentile(macro_vals, 100*alpha/2)),
                         float(np.percentile(macro_vals, 100*(1-alpha/2)))) if macro_vals else (float("nan"),)*2
    return out

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · operating_points                                             [B1]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : fija en VALIDACIÓN tres umbrales por etiqueta: "f1" (equilibrio), "cribado" (el más
#              alto que aún da Se≥sens_target) y "confirm" (el más bajo que aún da Sp≥spec_target).
# POR QUÉ    : un solo umbral no sirve en clínica. Cribar exige no perder enfermos; confirmar exige
#              no alarmar en falso. Son dos decisiones distintas sobre el mismo modelo.
# ENTRADAS   : probs/labels/mask de VALIDACIÓN · sens_target · spec_target
# SALIDAS    : dict {"f1"|"cribado"|"confirm": {etiqueta: umbral}}
# ORIGEN EDA : §11 · "Puntos de operación: fijar en VAL alta sensibilidad (cribado) y alta
#              especificidad (confirmación). Reportar Se/Sp/VPP/VPN".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              el VPP depende de la PREVALENCIA (13–36 % aquí): un VPP modesto puede valer para
#              cribar y ser inservible para confirmar. Discutir cada punto por su consecuencia clínica.
# ══════════════════════════════════════════════════════════════════════════════
def operating_points(probs, labels, mask, sens_target=0.90, spec_target=0.90):
    grid = np.linspace(0.01, 0.99, 99); pts = {"f1": {}, "cribado": {}, "confirm": {}}
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        if yt.sum() < 2 or (1 - yt).sum() < 2:
            for k in pts: pts[k][l] = 0.5
            continue
        best_f1, thr_f1 = -1, 0.5; thr_sens, thr_spec = grid[0], grid[-1]
        for t in grid:
            pred = (yp >= t).astype(float)
            tp = ((pred == 1) & (yt == 1)).sum(); fn = ((pred == 0) & (yt == 1)).sum()
            tn = ((pred == 0) & (yt == 0)).sum(); fp = ((pred == 1) & (yt == 0)).sum()
            f1 = f1_score(yt, pred, zero_division=0)
            if f1 > best_f1: best_f1, thr_f1 = f1, t
            if tp/max(tp+fn, 1) >= sens_target: thr_sens = max(thr_sens, t)
            if tn/max(tn+fp, 1) >= spec_target: thr_spec = min(thr_spec, t)
        pts["f1"][l], pts["cribado"][l], pts["confirm"][l] = float(thr_f1), float(thr_sens), float(thr_spec)
    return pts

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · clinical_report                                              [B1]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : evalúa unos umbrales y devuelve Se, Sp, VPP, VPN y la confusión por etiqueta.
# POR QUÉ    : AUC y AP resumen el ranking, pero la decisión se toma en UN umbral; el clínico
#              necesita saber cuántos enfermos se escapan y cuántas alarmas falsas se generan.
# ENTRADAS   : probs/labels/mask (TEST) · thresholds {etiqueta: umbral} · punto (nombre)
# SALIDAS    : DataFrame (punto, etiqueta, umbral, Se, Sp, VPP, VPN, TP/TN/FP/FN, prevalencia)
# ORIGEN EDA : §11 "Reportar Se/Sp/VPP/VPN" · §3 (la prevalencia condiciona el VPP).
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comparar FALSOS NEGATIVOS en cribado frente a FALSOS POSITIVOS en confirmación.
# ══════════════════════════════════════════════════════════════════════════════
def clinical_report(probs, labels, mask, thresholds, punto="f1"):
    rows = []
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        t = thresholds.get(l, 0.5); pred = (yp >= t).astype(float)
        tp = int(((pred == 1) & (yt == 1)).sum()); tn = int(((pred == 0) & (yt == 0)).sum())
        fp = int(((pred == 1) & (yt == 0)).sum()); fn = int(((pred == 0) & (yt == 1)).sum())
        rows.append({"punto": punto, "etiqueta": l, "umbral": round(t, 3),
                     "Se": tp/max(tp+fn,1), "Sp": tn/max(tn+fp,1), "VPP": tp/max(tp+fp,1), "VPN": tn/max(tn+fn,1),
                     "TP": tp, "TN": tn, "FP": fp, "FN": fn, "prevalencia": (tp+fn)/max(tp+tn+fp+fn,1)})
    return pd.DataFrame(rows)

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · calibration_report                                           [B2]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : Brier score por etiqueta + puntos de la curva de fiabilidad (10 bins por cuantiles).
# POR QUÉ    : la herramienta clínica muestra PROBABILIDADES; si no están calibradas, un 0,8 no
#              significa "80 % de estos pacientes lo tienen" y la cifra engaña al médico.
# ENTRADAS   : probs/labels/mask · n_bins
# SALIDAS    : (DataFrame Brier por etiqueta, dict {etiqueta:(frac_obs, media_pred)})
# ORIGEN EDA : §11 · "Calibración en VAL; verificar con Brier score y curva de fiabilidad;
#              recomprobar tras fusión".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comparar Brier ANTES vs DESPUÉS; si no mejora, decirlo. RECALIBRAR tras la fusión.
# ══════════════════════════════════════════════════════════════════════════════
def calibration_report(probs, labels, mask, n_bins=10):
    rows, curves = [], {}
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        if len(np.unique(yt)) < 2: rows.append({"etiqueta": l, "Brier": float("nan")}); continue
        rows.append({"etiqueta": l, "Brier": float(brier_score_loss(yt, np.clip(yp, 0, 1)))})
        try: curves[l] = calibration_curve(yt, np.clip(yp, 0, 1), n_bins=n_bins, strategy="quantile")
        except Exception: curves[l] = (np.array([]), np.array([]))
    return pd.DataFrame(rows), curves

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · consistency_no_finding                                       [B6]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : correlación entre P(«Sin hallazgo») y max P(patología), y % de casos en que ambas
#              superan 0,5 a la vez (el modelo se contradice).
# POR QUÉ    : las 6 cabezas son independientes; nada las obliga a ser coherentes. Un modelo que
#              afirma "sano" y "con derrame" a la vez es inaceptable en una herramienta clínica.
# ENTRADAS   : probs (N,6)
# SALIDAS    : dict {correlación (debe ser NEGATIVA), % incoherentes}
# ORIGEN EDA : §3 · "«Sin hallazgo» se modela como una etiqueta más (P(normal) útil como chequeo de
#              consistencia), nunca como fuente de negativos".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              correlación ~0 o positiva ⇒ la cabeza «Sin hallazgo» no aprende normalidad → no
#              mostrarla en la herramienta o revisarla.
# ══════════════════════════════════════════════════════════════════════════════
def consistency_no_finding(probs):
    j_nf = LABELS.index(NO_FINDING); j_p = [j for j in range(N_LABELS) if j != j_nf]
    p_nf, p_max = probs[:, j_nf], probs[:, j_p].max(axis=1)
    return {"corr_NoFinding_vs_maxPatologia": float(np.corrcoef(p_nf, p_max)[0, 1]),
            "pct_incoherentes": float(((p_nf > 0.5) & (p_max > 0.5)).mean()*100)}

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · flag_importance_share                                        [B7]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : % de la importancia total que recae en los flags de missingness ("falta:...").
# POR QUÉ    : la ausencia es MNAR (a los graves se les piden más pruebas). Es señal útil, pero si el
#              modelo se apoya demasiado aprende "le hicieron analítica ⇒ está enfermo", no biología.
# ENTRADAS   : imp (vector de importancias) · names (nombres de features)
# SALIDAS    : dict {% importancia en flags, flag más influyente}
# ORIGEN EDA : §6 · "vigilar que el modelo no dependa en exceso del patrón de ausencia".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              si supera ~25 %, advertir de que parte del rendimiento es un ARTEFACTO del proceso
#              asistencial y NO generalizaría a otro hospital con otra política de peticiones.
# ══════════════════════════════════════════════════════════════════════════════
def flag_importance_share(imp, names):
    imp = np.asarray(imp, float); tot = imp.sum() + 1e-12
    idx = [i for i, n in enumerate(names) if str(n).startswith("falta:")]
    if not idx: return {"pct_importancia_flags": 0.0, "flag_top": None}
    return {"pct_importancia_flags": float(imp[idx].sum()/tot*100),
            "flag_top": names[idx[int(np.argmax(imp[idx]))]]}

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · stratified_report                                            [B8]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : recalcula la métrica primaria dentro de cada subgrupo (sexo, etnia, ingreso) y por
#              PROYECCIÓN radiográfica (cxr_view).
# POR QUÉ    : (a) equidad; (b) robustez — la placa AP se hace al paciente encamado y magnifica la
#              silueta cardíaca, así que conviene ver si el rendimiento depende de la proyección.
# ENTRADAS   : df (metadatos del split) · probs/labels/mask · cols
# SALIDAS    : DataFrame (variable, grupo, n, macro_AP, macro_AUC)
# ORIGEN EDA : §2/§9 "evaluar equidad por sexo y etnia" · ANEXO CXR "monitorizar el efecto de cxr_view".
# DECISIÓN DE DISEÑO: cxr_view se usa SOLO aquí. NUNCA como predictor: es proxy de gravedad y su
#              inclusión inflaría el resultado por un atajo asistencial en lugar de señal biológica.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              subgrupos con n<60 pueden diferir por PURO RUIDO; no afirmar inequidad sin IC.
# ══════════════════════════════════════════════════════════════════════════════
def stratified_report(df, probs, labels, mask, cols=("gender", "race", "admission_type", "cxr_view")):
    d = df.reset_index(drop=True); rows = []
    for col in cols:
        if col not in d.columns: continue
        for v in sorted(d[col].dropna().unique(), key=str):
            idx = d.index[d[col] == v].to_numpy()
            if len(idx) < 15:
                rows.append({"variable": col, "grupo": str(v), "n": len(idx), "macro_AP": np.nan, "macro_AUC": np.nan}); continue
            mm = multilabel_metrics(probs[idx], labels[idx], mask[idx])
            rows.append({"variable": col, "grupo": str(v), "n": len(idx),
                         "macro_AP": mm["macro_AP_path"], "macro_AUC": mm["macro_AUC_path"]})
    return pd.DataFrame(rows)

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · importancia_mi_anova                                         [B5]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : importancia por Información Mutua y por F de ANOVA, promediada sobre las etiquetas.
# POR QUÉ    : el Gini de los árboles se SESGA con clases desbalanceadas y con variables de muchos
#              cortes; MI y ANOVA son criterios independientes que permiten validar el ranking.
# ENTRADAS   : X (N,F) · y (N,6) · m (N,6) · names
# SALIDAS    : DataFrame ordenado (variable, importancia_MI, importancia_ANOVA, media)
# ORIGEN EDA : §10 · "en Opacidad, fiarse más de MI/ANOVA que de RF-Gini por el desbalanceo".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              donde Gini y MI/ANOVA coincidan, la conclusión es sólida; donde discrepen (esperable
#              en Lung Opacity y en los flags de gasometría), primar MI/ANOVA y explicarlo.
# ══════════════════════════════════════════════════════════════════════════════
def importancia_mi_anova(X, y, m, names, seed=SEED):
    Xi = np.nan_to_num(np.where(np.isnan(X), np.nanmedian(X, axis=0), X), nan=0.0, posinf=0.0, neginf=0.0)
    acc_mi = np.zeros(Xi.shape[1]); acc_f = np.zeros(Xi.shape[1]); n_ok = 0
    for j in range(N_LABELS):
        sel = m[:, j] == 1; yj = y[sel, j]
        if len(np.unique(yj)) < 2: continue
        mi = mutual_info_classif(Xi[sel], yj, random_state=seed)
        fv = np.nan_to_num(f_classif(Xi[sel], yj)[0], nan=0.0, posinf=0.0)
        acc_mi += mi/(mi.max()+1e-12); acc_f += fv/(fv.max()+1e-12); n_ok += 1
    acc_mi /= max(n_ok, 1); acc_f /= max(n_ok, 1)
    return (pd.DataFrame({"variable": names, "importancia_MI": np.round(acc_mi, 4),
                          "importancia_ANOVA": np.round(acc_f, 4)})
            .assign(media=lambda d: (d.importancia_MI + d.importancia_ANOVA)/2)
            .sort_values("media", ascending=False).reset_index(drop=True))

print("Kit clínico listo: B1 puntos de operación · B2 calibración · B4 IC bootstrap · B5 MI/ANOVA ·")
print("                   B6 consistencia · B7 flags MNAR · B8 estratificación (incl. cxr_view)")

In [ ]:
# CELDA 8 · MLP PROFUNDO multilabel enmascarado
# ══ MaskedBCE ════════════════════════════════════════════════════════════════
# QUÉ HACE: entropía cruzada binaria por etiqueta, ponderada con pos_weight y MULTIPLICADA por la
#           máscara (los −1 no contribuyen al gradiente). Opción focal.
# FINALIDAD: implementar el U-Ignore a nivel de pérdida y compensar el desbalanceo por etiqueta.
# ORIGEN EDA: §3 · recuadros naranjas "el −1 se enmascara (fuera de pérdida y métrica)" y
#            "desbalanceo moderado 1,7:1–6,3:1 → pos_weight = n_neg/n_pos en la BCE. Nunca SMOTE".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): comprobar si la variante focal aporta sobre la BCE
#            simple; con desbalanceo solo moderado el EDA anticipa que la ganancia será pequeña.
class MaskedBCE(nn.Module):
    def __init__(s,pos_weight,focal=False,gamma=2.0):
        super().__init__(); s.register_buffer("pw",pos_weight); s.focal=focal; s.g=gamma
    def forward(s,logits,labels,mask):
        pw=s.pw.to(logits.device)
        bce=F.binary_cross_entropy_with_logits(logits,labels,pos_weight=pw,reduction="none")
        if s.focal:
            p=torch.sigmoid(logits); pt=labels*p+(1-labels)*(1-p); bce=((1-pt)**s.g)*bce
        bce=bce*mask; return bce.sum()/mask.sum().clamp(min=1e-8)

# ══ TabMLP ═══════════════════════════════════════════════════════════════════
# QUÉ HACE: MLP de 3 bloques con BatchNorm+Dropout y 6 salidas (una por etiqueta, sin softmax).
# FINALIDAD: modelar las 6 etiquetas como binarias INDEPENDIENTES (binary relevance).
# ORIGEN EDA: §1/§3 · recuadros naranjas "sigmoides independientes, no softmax: las patologías no son
#            mutuamente excluyentes" y "Jaccard ≤ 0,30 → no hace falta modelar dependencias".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): si la fusión mejora mucho más que el mono-modelo,
#            es señal de que la dependencia entre etiquetas se captura mejor entre modalidades.
class TabMLP(nn.Module):
    def __init__(s,d,hidden=512,dropout=0.3):
        super().__init__(); h2,h3=hidden//2,hidden//4
        s.net=nn.Sequential(nn.Linear(d,hidden),nn.BatchNorm1d(hidden),nn.ReLU(inplace=True),nn.Dropout(dropout),
                            nn.Linear(hidden,h2),nn.BatchNorm1d(h2),nn.ReLU(inplace=True),nn.Dropout(dropout),
                            nn.Linear(h2,h3),nn.BatchNorm1d(h3),nn.ReLU(inplace=True),nn.Dropout(dropout*0.6),
                            nn.Linear(h3,N_LABELS))
    def forward(s,x): return s.net(x)

# ══ dyn_pos_weight ═══════════════════════════════════════════════════════════
# QUÉ HACE: calcula pos_weight = n_neg/n_pos POR etiqueta sobre los objetivos EFECTIVOS del fold.
# FINALIDAD: recalcular el balanceo en cada fold en lugar de fijar constantes globales.
# ORIGEN EDA: §3 · recuadro naranja "pos_weight (train): 2,28/1,85/3,43/2,15/6,31/1,66. Recalcular
#            POR FOLD en validación cruzada".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): con el etiquetado FINAL (NaN→0) los pesos deben
#            coincidir con los del EDA; si se desvían, hay un problema en build_targets.
def dyn_pos_weight(y,m,clip=10.0):
    w=np.ones(N_LABELS,np.float32)
    for j in range(N_LABELS):
        s=m[:,j]==1; pos=(y[s,j]==1).sum(); neg=(y[s,j]==0).sum(); w[j]=np.clip(neg/max(pos,1),1/clip,clip)
    return torch.tensor(w,dtype=torch.float32)

# ══ train_mlp ════════════════════════════════════════════════════════════════
# QUÉ HACE: entrena el MLP con early-stopping sobre VALIDACIÓN.
# FINALIDAD: parar en el mejor punto según la MÉTRICA PRIMARIA = AUC-PR macro (antes era AUC-ROC).
# ORIGEN EDA: §3 · recuadro naranja "métrica primaria AUC-PR; con desbalanceo la AP es más honesta".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): el cambio de criterio de parada puede seleccionar
#            una época distinta a la del criterio ROC → documentar si cambia el modelo elegido.
def train_mlp(Xtr,ytr,mtr,Xva,yva,mva,cfg,epochs=150,patience=15):
    crit=MaskedBCE(dyn_pos_weight(ytr,mtr),focal=cfg.get("focal",False)).to(DEVICE)
    model=TabMLP(Xtr.shape[1],cfg.get("hidden",512),cfg.get("dropout",0.3)).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.get("lr",8e-4),weight_decay=cfg.get("wd",1e-4))
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs,eta_min=1e-7)
    Xt=torch.tensor(Xtr); Yt=torch.tensor(ytr); Mt=torch.tensor(mtr); n=len(Xt); bs=256
    best,bs_state,wait=-1.0,None,0
    for ep in range(epochs):
        model.train(); perm=torch.randperm(n)
        for i in range(0,n,bs):
            idx=perm[i:i+bs]
            if len(idx)<2: continue
            opt.zero_grad(); loss=crit(model(Xt[idx].to(DEVICE)),Yt[idx].to(DEVICE),Mt[idx].to(DEVICE))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        sched.step()
        with torch.no_grad():
            model.eval(); pv=torch.sigmoid(model(torch.tensor(Xva).to(DEVICE))).cpu().numpy()
        sc=multilabel_metrics(pv,yva,mva)["macro_AP_path"]     # ← PRIMARIA (antes macro_AUC_path)
        if not np.isnan(sc) and sc>best+1e-4: best,bs_state,wait=sc,copy.deepcopy(model.state_dict()),0
        else:
            wait+=1
            if wait>=patience: break
    if bs_state is not None: model.load_state_dict(bs_state)
    with torch.no_grad():
        model.eval(); pv=torch.sigmoid(model(torch.tensor(Xva).to(DEVICE))).cpu().numpy()
    return model, pv
print("MLP listo (early-stopping por AUC-PR).")

In [ ]:
# CELDA 9 · One-vs-rest + modelos (con config por modelo)
def ovr_fit_predict(make_clf, Xtr, ytr, mtr, Xva, use_spw=True):
    probs=np.full((len(Xva),N_LABELS),0.5,np.float32)
    for j in range(N_LABELS):
        sel=mtr[:,j]==1; Xj=Xtr[sel]; yj=ytr[sel,j]
        if len(np.unique(yj))<2:
            probs[:,j]=float(yj.mean()) if len(yj) else 0.5; continue
        spw=(yj==0).sum()/max((yj==1).sum(),1)
        clf=make_clf(spw if use_spw else None); clf.fit(Xj,yj); probs[:,j]=clf.predict_proba(Xva)[:,1]
    return probs
def make_xgb(spw,cfg=None):
    p=dict(n_estimators=800,max_depth=5,learning_rate=0.03,subsample=0.8,colsample_bytree=0.8,min_child_weight=2,reg_lambda=1.5,tree_method="hist",eval_metric="logloss",n_jobs=4,random_state=SEED)
    if cfg: p.update(cfg)
    return xgb.XGBClassifier(scale_pos_weight=spw,**p)
def make_lgb(spw,cfg=None):
    p=dict(n_estimators=800,num_leaves=63,learning_rate=0.03,subsample=0.8,colsample_bytree=0.8,min_child_samples=20,reg_lambda=1.5,n_jobs=4,random_state=SEED,verbosity=-1)
    if cfg: p.update(cfg)
    return lgb.LGBMClassifier(scale_pos_weight=spw,**p)
# ══ NOTA DE MEMORIA (equipo CPU-only con RAM ajustada) ═══════════════════════
# n_jobs>1 en scikit-learn usa joblib/loky, que lanza SUBPROCESOS de Python completos; cada uno
# re-importa torch/xgboost/lightgbm (~400 MB). Con poca RAM libre el kernel MUERE sin traceback
# ('DeadKernelError: Kernel died'). Por eso los estimadores de sklearn van con n_jobs=1.
# XGBoost y LightGBM usan HILOS OpenMP (memoria compartida) → mantienen n_jobs=4 sin riesgo.
# Esto NO altera los resultados, solo el tiempo de cómputo.
def make_logreg(spw,cfg=None):
    # n_jobs=1 A PROPOSITO: evita que joblib/loky lance subprocesos y agote la RAM.
    return LogisticRegression(max_iter=3000,class_weight="balanced",C=(cfg or {}).get("C",1.0),n_jobs=1)
def make_tabpfn(spw=None,n_ens=4):
    return TabPFNClassifier()   # cliente NUBE (token en celda 2). NO incluyas tu clave aquí.

def predict_model(name, tr, va, Xd_tr, Xd_va, cfgs):
    cfg=cfgs.get(name,{})
    if name=="XGBoost":  return ovr_fit_predict(lambda spw: make_xgb(spw,cfg), TREE_tr[tr], y_train[tr], m_train[tr], TREE_tr[va])
    if name=="LightGBM": return ovr_fit_predict(lambda spw: make_lgb(spw,cfg), TREE_tr[tr], y_train[tr], m_train[tr], TREE_tr[va])
    if name=="LogReg":   return ovr_fit_predict(lambda spw: make_logreg(spw,cfg), Xd_tr, y_train[tr], m_train[tr], Xd_va, use_spw=False)
    if name=="MLP":
        c=cfg if cfg else {"lr":8e-4,"dropout":0.3}; _,pv=train_mlp(Xd_tr,y_train[tr],m_train[tr],Xd_va,y_train[va],m_train[va],c); return pv
    if name=="TabPFN":
        if len(tr)<=TABPFN_MAX: Xt,yt,mt=Xd_tr,y_train[tr],m_train[tr]
        else:
            pos=np.random.RandomState(SEED).choice(len(tr),TABPFN_MAX,replace=False); Xt,yt,mt=Xd_tr[pos],y_train[tr][pos],m_train[tr][pos]
        return ovr_fit_predict(lambda spw: make_tabpfn(spw), Xt, yt, mt, Xd_va, use_spw=False)
    raise ValueError(name)
print("Modelos listos.")


In [ ]:
# CELDA 10 · TUNING POR MODELO (Optuna en cada arquitectura; complejo justo)
# ══ cv_macro ═════════════════════════════════════════════════════════════════
# QUÉ HACE: validación cruzada interna que devuelve la MÉTRICA PRIMARIA (AUC-PR macro) de una config.
# FINALIDAD: que Optuna optimice la métrica correcta para datos desbalanceados (antes optimizaba ROC).
# ORIGEN EDA: §3 · recuadro naranja "métrica primaria AUC-PR + IC bootstrap; ROC secundaria".
# NOTA sin fuga: la imputación y el escalado ROBUSTO se ajustan DENTRO de cada fold (§6 EDA).
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): los hiperparámetros elegidos pueden diferir de los
#            de la versión anterior (que optimizaba ROC) → indicar el cambio al comparar resultados.
def cv_macro(predict_cfg, k=3):
    kf=KFold(k,shuffle=True,random_state=SEED); sc=[]
    for tr,va in kf.split(np.arange(len(df_train))):
        imp,scl=dense_fit(tr); Xd_tr=dense_tx(imp,scl,DENSE_tr[tr]); Xd_va=dense_tx(imp,scl,DENSE_tr[va])
        sc.append(multilabel_metrics(predict_cfg(tr,va,Xd_tr,Xd_va),y_train[va],m_train[va])["macro_AP_path"])
    return float(np.nanmean(sc))
CFGS={}
def tune(name,space,n_trials,builder_tree=None,dense=False):
    def obj(trial):
        cfg=space(trial)
        if dense:
            if name=="LogReg":
                pc=lambda tr,va,Xd_tr,Xd_va: ovr_fit_predict(lambda spw: make_logreg(spw,cfg),Xd_tr,y_train[tr],m_train[tr],Xd_va,use_spw=False)
            else:
                def pc(tr,va,Xd_tr,Xd_va):
                    _,pv=train_mlp(Xd_tr,y_train[tr],m_train[tr],Xd_va,y_train[va],m_train[va],cfg); return pv
        else:
            pc=lambda tr,va,Xd_tr,Xd_va: ovr_fit_predict(lambda spw: builder_tree(spw,cfg),TREE_tr[tr],y_train[tr],m_train[tr],TREE_tr[va])
        return cv_macro(pc)
    st=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=SEED)); st.optimize(obj,n_trials=n_trials,show_progress_bar=True)
    CFGS[name]=st.best_params; print(f"   {name}: CV macro_AP={st.best_value:.4f} cfg={st.best_params}")

print("Tuning XGBoost..."); tune("XGBoost",lambda t:dict(n_estimators=t.suggest_int("n_estimators",400,1200,step=100),max_depth=t.suggest_int("max_depth",3,8),learning_rate=t.suggest_float("learning_rate",0.01,0.1,log=True),subsample=t.suggest_float("subsample",0.6,1.0),colsample_bytree=t.suggest_float("colsample_bytree",0.5,1.0),min_child_weight=t.suggest_int("min_child_weight",1,8),reg_lambda=t.suggest_float("reg_lambda",0.1,8.0,log=True)),TUNE_GBM,builder_tree=make_xgb)
print("Tuning LightGBM..."); tune("LightGBM",lambda t:dict(n_estimators=t.suggest_int("n_estimators",400,1200,step=100),num_leaves=t.suggest_int("num_leaves",15,200),learning_rate=t.suggest_float("learning_rate",0.01,0.1,log=True),subsample=t.suggest_float("subsample",0.6,1.0),colsample_bytree=t.suggest_float("colsample_bytree",0.5,1.0),min_child_samples=t.suggest_int("min_child_samples",10,60),reg_lambda=t.suggest_float("reg_lambda",0.1,8.0,log=True)),TUNE_GBM,builder_tree=make_lgb)
print("Tuning LogReg..."); tune("LogReg",lambda t:{"C":t.suggest_float("C",0.01,10,log=True)},TUNE_LOGREG,dense=True)
print("Tuning MLP..."); tune("MLP",lambda t:{"lr":t.suggest_float("lr",3e-4,3e-3,log=True),"dropout":t.suggest_float("dropout",0.1,0.5),"hidden":t.suggest_categorical("hidden",[256,512]),"focal":t.suggest_categorical("focal",[False,True]),"wd":t.suggest_float("wd",1e-5,1e-3,log=True)},TUNE_MLP,dense=True)
if USE_TABPFN: CFGS["TabPFN"]={}
print("Tuning terminado (objetivo = AUC-PR macro). CFGS:",list(CFGS))

In [ ]:
# CELDA 11 · COMPARATIVA K-FOLD (configs tuneadas) + ENSEMBLE (promedio) + boxplot
# SELECCIÓN DEL MEJOR = por AUC-PR macro (primaria). Se conserva el AUC-ROC como referencia.
# ORIGEN EDA: §3 (AP primaria) · §1 (IC bootstrap por el tamaño reducido de val/test).
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): el "mejor" bajo AP puede NO ser el mismo que bajo
#            ROC; si cambia, explicarlo (la AP prioriza precisión en la clase positiva minoritaria).
BASE_MODELS=["XGBoost","LightGBM","LogReg","MLP"] + (["TabPFN"] if USE_TABPFN else [])
oof={n:np.zeros((len(df_train),N_LABELS),np.float32) for n in BASE_MODELS+["Ensemble"]}
fold_ap={n:[] for n in BASE_MODELS+["Ensemble"]}; fold_auc={n:[] for n in BASE_MODELS+["Ensemble"]}
perlabel_ap={n:{l:[] for l in LABELS} for n in BASE_MODELS+["Ensemble"]}
failed=set(); kf=KFold(K_COMPARE,shuffle=True,random_state=SEED); t0=time.time()
for k,(tr,va) in enumerate(kf.split(np.arange(len(df_train)))):
    print(f"\n— Fold {k+1}/{K_COMPARE} —")
    imp,sc=dense_fit(tr); Xd_tr=dense_tx(imp,sc,DENSE_tr[tr]); Xd_va=dense_tx(imp,sc,DENSE_tr[va])
    preds_this=[]
    for n in BASE_MODELS:
        if n in failed: continue
        try:
            tk=time.time(); pv=predict_model(n,tr,va,Xd_tr,Xd_va,CFGS); oof[n][va]=pv; preds_this.append(pv)
            mm=multilabel_metrics(pv,y_train[va],m_train[va]); fold_ap[n].append(mm["macro_AP_path"]); fold_auc[n].append(mm["macro_AUC_path"])
            for l in LABELS: perlabel_ap[n][l].append(mm[l]["AP"])
            print(f"   {n:9s} macroAP={mm['macro_AP_path']:.4f} (AUC={mm['macro_AUC_path']:.4f}) ({(time.time()-tk)/60:.1f} min)")
        except Exception as ex:
            print(f"   ⚠ {n} FALLA y se OMITE: {type(ex).__name__}: {str(ex)[:120]}"); failed.add(n)
    if preds_this:
        ens=np.mean(preds_this,axis=0); oof["Ensemble"][va]=ens
        mm=multilabel_metrics(ens,y_train[va],m_train[va]); fold_ap["Ensemble"].append(mm["macro_AP_path"]); fold_auc["Ensemble"].append(mm["macro_AUC_path"])
        for l in LABELS: perlabel_ap["Ensemble"][l].append(mm[l]["AP"])
        print(f"   {'Ensemble':9s} macroAP={mm['macro_AP_path']:.4f} (AUC={mm['macro_AUC_path']:.4f})")
    gc.collect()
MODELS=[n for n in BASE_MODELS if n not in failed]
print(f"\nComparativa en {(time.time()-t0)/60:.1f} min · Modelos válidos: {MODELS} (+Ensemble)")
ALL=MODELS+["Ensemble"]
print(f"\n{'Modelo':10s} {'macroAP (CV)':>16} {'macroAUC (CV)':>16}"); print("-"*44); rank=[]
for n in ALL:
    print(f"{n:10s} {np.mean(fold_ap[n]):.4f} ± {np.std(fold_ap[n]):.4f}   {np.mean(fold_auc[n]):.4f}")
    rank.append((n,float(np.mean(fold_ap[n]))))
rank.sort(key=lambda x:-x[1]); BEST=rank[0][0]; print(f"\n>>> Mejor (por AUC-PR): {BEST} (macroAP={rank[0][1]:.4f})")
fig,ax=plt.subplots(figsize=(11,5))
data=[[a for l in PATHOLOGY_LABELS for a in perlabel_ap[n][l] if not np.isnan(a)] for n in ALL]
bp=ax.boxplot(data,labels=ALL,patch_artist=True)
for i,patch in enumerate(bp["boxes"]): patch.set_facecolor("#f1c40f" if ALL[i]=="Ensemble" else "#7fb3d5")
ax.set_ylabel("AUC-PR (patologías, CV)"); ax.set_title("LABS v2 — Comparativa de arquitecturas + Ensemble (AUC-PR)")
plt.tight_layout(); plt.savefig(FIG_DIR/"boxplot_ap_modelos.png",dpi=150,bbox_inches="tight"); plt.show()
print("Boxplot guardado.")

In [ ]:
# CELDA 12 · MATRICES DE CONFUSIÓN POR MODELO (OOF, umbral 0.5)
cols_show=MODELS+["Ensemble"]
fig,axes=plt.subplots(1,len(cols_show),figsize=(3.0*len(cols_show),3.4))
if len(cols_show)==1: axes=[axes]
for ax,n in zip(axes,cols_show):
    yt_all=[]; yp_all=[]
    for j in range(N_LABELS):
        s=m_train[:,j]==1; yt_all.append(y_train[s,j]); yp_all.append((oof[n][s,j]>=0.5).astype(int))
    cm=confusion_matrix(np.concatenate(yt_all),np.concatenate(yp_all),labels=[0,1])
    sns.heatmap(cm,annot=True,fmt="d",cmap="Purples",cbar=False,ax=ax,xticklabels=["P0","P1"],yticklabels=["R0","R1"])
    ax.set_title(n,fontsize=10)
fig.suptitle("v2.2 — Confusión global por modelo (OOF)",fontweight="bold")
plt.tight_layout(); plt.savefig(FIG_DIR/"confusion_por_modelo.png",dpi=150,bbox_inches="tight"); plt.show()
print("Guardadas.")


In [ ]:
# CELDA 13 · MODELO FINAL + CALIBRACIÓN + PUNTOS DE OPERACIÓN + PRE-REGISTRO + TEST (una sola vez)
# Orden del protocolo (§11 EDA): entrenar en TRAIN → calibrar y fijar umbrales en VAL → pre-registrar
# → evaluar TEST UNA vez. Nada se reajusta después de mirar test.
from sklearn.isotonic import IsotonicRegression

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · final_one
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : reentrena la arquitectura indicada sobre TODO el train y predice (test, val).
# POR QUÉ    : tras elegir el ganador por AUC-PR en CV, el modelo definitivo debe usar todos los datos.
# ENTRADAS   : name (nombre de la arquitectura)
# SALIDAS    : (probs_test, probs_val)
# ORIGEN EDA : §11 · "TRAIN entrena; VAL calibra+umbrales+selección; TEST una sola vez".
# ══════════════════════════════════════════════════════════════════════════════
imp_f,sc_f=dense_fit(np.arange(len(df_train)))
Xd_tr_f=dense_tx(imp_f,sc_f,DENSE_tr); Xd_te=dense_tx(imp_f,sc_f,DENSE_te); Xd_vl=dense_tx(imp_f,sc_f,DENSE_vl)
def final_one(name):
    cfg=CFGS.get(name,{})
    if name=="XGBoost": return (ovr_fit_predict(lambda spw: make_xgb(spw,cfg),TREE_tr,y_train,m_train,TREE_te), ovr_fit_predict(lambda spw: make_xgb(spw,cfg),TREE_tr,y_train,m_train,TREE_vl))
    if name=="LightGBM": return (ovr_fit_predict(lambda spw: make_lgb(spw,cfg),TREE_tr,y_train,m_train,TREE_te), ovr_fit_predict(lambda spw: make_lgb(spw,cfg),TREE_tr,y_train,m_train,TREE_vl))
    if name=="LogReg": return (ovr_fit_predict(lambda spw: make_logreg(spw,cfg),Xd_tr_f,y_train,m_train,Xd_te,use_spw=False), ovr_fit_predict(lambda spw: make_logreg(spw,cfg),Xd_tr_f,y_train,m_train,Xd_vl,use_spw=False))
    if name=="MLP":
        c=cfg if cfg else {"lr":8e-4,"dropout":0.3}; mdl,pt=train_mlp(Xd_tr_f,y_train,m_train,Xd_vl,y_val,m_val,c)
        with torch.no_grad():
            mdl.eval(); ptst=torch.sigmoid(mdl(torch.tensor(Xd_te).to(DEVICE))).cpu().numpy()
        return ptst,pt
    if name=="TabPFN":
        sub=np.arange(len(df_train)) if len(df_train)<=TABPFN_MAX_FINAL else np.random.RandomState(SEED).choice(len(df_train),TABPFN_MAX_FINAL,replace=False)
        return (ovr_fit_predict(lambda spw: make_tabpfn(spw),Xd_tr_f[sub],y_train[sub],m_train[sub],Xd_te,use_spw=False), ovr_fit_predict(lambda spw: make_tabpfn(spw),Xd_tr_f[sub],y_train[sub],m_train[sub],Xd_vl,use_spw=False))
    raise ValueError(name)
if BEST=="Ensemble":
    te_list=[]; vl_list=[]
    for n in MODELS:
        try: t_,v_=final_one(n); te_list.append(t_); vl_list.append(v_)
        except Exception as ex: print(f"   {n} omitido en ensemble final: {ex}")
    test_pred_raw=np.mean(te_list,axis=0); val_pred_raw=np.mean(vl_list,axis=0)
else:
    test_pred_raw,val_pred_raw=final_one(BEST)

# ── CALIBRACIÓN ISOTÓNICA ajustada en VAL ─────────────────────────────────────────────────────
# ORIGEN EDA §11: las probabilidades deben ser interpretables por el clínico (un 0,8 ≈ 80 %).
calibrators={}
for j,l in enumerate(LABELS):
    s=m_val[:,j]==1; yt=y_val[s,j]; yp=val_pred_raw[s,j]
    calibrators[l]=None if len(np.unique(yt))<2 else IsotonicRegression(out_of_bounds="clip").fit(yp,yt)
def apply_cal(P):
    O=P.copy()
    for j,l in enumerate(LABELS):
        if calibrators[l] is not None: O[:,j]=calibrators[l].predict(P[:,j])
    return O
val_pred=apply_cal(val_pred_raw); test_pred=apply_cal(test_pred_raw)

# ── B2 · ¿mejora realmente la calibración? Brier antes vs después (en VAL) ─────────────────────
brier_pre,_      = calibration_report(val_pred_raw,y_val,m_val)
brier_post,curvas= calibration_report(val_pred,    y_val,m_val)
cal_cmp=brier_pre.merge(brier_post,on="etiqueta",suffixes=("_sin_calibrar","_calibrado"))
cal_cmp["mejora"]=cal_cmp["Brier_sin_calibrar"]-cal_cmp["Brier_calibrado"]
print("── B2 · Brier en VALIDACIÓN (menor = mejor) ──")
print(cal_cmp.to_string(index=False,float_format=lambda v:f"{v:.4f}"))
print(f"   Mejora en {(cal_cmp['mejora']>0).sum()}/{len(cal_cmp)} etiquetas")
cal_cmp.to_csv(OUTPUT_DIR/"calibracion_brier.csv",index=False)
fig,ax=plt.subplots(figsize=(6.4,6)); ax.plot([0,1],[0,1],"--",color="gray",lw=1,label="calibración perfecta")
for l in LABELS:
    fr,mp=curvas.get(l,(np.array([]),np.array([])))
    if len(fr): ax.plot(mp,fr,"o-",ms=4,lw=1.6,label=l)
ax.set_xlabel("probabilidad predicha media"); ax.set_ylabel("frecuencia observada")
ax.set_title("Curva de fiabilidad tras calibración (VAL)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR/"curva_fiabilidad.png",dpi=150,bbox_inches="tight"); plt.show()

# ── B1 · Tres puntos de operación fijados en VAL ──────────────────────────────────────────────
PUNTOS=operating_points(val_pred,y_val,m_val,sens_target=0.90,spec_target=0.90)
thr_val=PUNTOS["f1"]
print("\n── B1 · Umbrales por punto de operación (fijados en VAL) ──")
print(f"{'Etiqueta':18s} {'F1':>6} {'Cribado(Se>=.90)':>18} {'Confirm(Sp>=.90)':>18}")
for l in LABELS: print(f"{l:18s} {PUNTOS['f1'][l]:6.2f} {PUNTOS['cribado'][l]:18.2f} {PUNTOS['confirm'][l]:18.2f}")

# ── B3 · PRE-REGISTRO de la regla (antes de tocar TEST) ───────────────────────────────────────
json.dump({"modelo":f"LABS v2 · {BEST}","etiquetado":"POS=(==1); NEG=(==0)|(NaN->0); -1 ENMASCARADO",
           "metrica_primaria":"AUC-PR (macro_AP_path); ROC secundaria; IC bootstrap 1000",
           "calibracion":"isotonica ajustada en VAL","cfgs":CFGS,
           "puntos_operacion":{"f1":PUNTOS["f1"],"cribado_Se>=0.90":PUNTOS["cribado"],"confirmacion_Sp>=0.90":PUNTOS["confirm"]},
           "cxr_view":"EXCLUIDA como predictor (proxy de gravedad); solo para estratificar",
           "nota":"TEST se evalua UNA sola vez con estos umbrales."},
          open(OUTPUT_DIR/"preregistro_regla_decision.json","w",encoding="utf-8"),indent=2,default=str,ensure_ascii=False)
print("\n── B3 · Regla PRE-REGISTRADA (test aún sin tocar)")

# ── EVALUACIÓN EN TEST · discriminación con IC (B4) ───────────────────────────────────────────
M=multilabel_metrics(test_pred,y_test,m_test,thresholds=thr_val)
ci_ap =bootstrap_ci_metric(test_pred,y_test,m_test,metric="ap")
ci_auc=bootstrap_ci_metric(test_pred,y_test,m_test,metric="auc")
print(f"\nMEJOR = {BEST}   (métrica primaria: AUC-PR)")
print(f"{'Etiqueta':18s} {'AP':>7} {'IC95% AP':>16} {'prev':>6} {'AUC':>7} {'IC95% AUC':>16} {'N+':>5} {'N-':>5}")
print("-"*96); rows=[]
for l in LABELS:
    m=M[l]; auc=f"{m['AUC']:.4f}" if not np.isnan(m['AUC']) else "  N/A"
    la,ha=ci_ap[l]; lu,hu=ci_auc[l]
    fa=f"[{la:.3f},{ha:.3f}]" if not np.isnan(la) else "        N/A"
    fu=f"[{lu:.3f},{hu:.3f}]" if not np.isnan(lu) else "        N/A"
    print(f"{l:18s} {m['AP']:7.4f} {fa:>16} {m['prevalencia']:6.3f} {auc:>7} {fu:>16} {m['n_pos']:5d} {m['n_neg']:5d}")
    rows.append({"label":l,**{k:m[k] for k in ["AUC","AP","F1","sens","spec","n_pos","n_neg","prevalencia"]},
                 "AP_ci_lo":la,"AP_ci_hi":ha,"AUC_ci_lo":lu,"AUC_ci_hi":hu,
                 "AP_supera_prevalencia":bool(m["AP"]>m["prevalencia"])})
print("-"*96)
mlo,mhi=ci_ap["macro_path"]; ulo,uhi=ci_auc["macro_path"]
print(f"MACRO patol.: AP={M['macro_AP_path']:.4f} IC95%=[{mlo:.3f},{mhi:.3f}] (PRIMARIA)")
print(f"              AUC={M['macro_AUC_path']:.4f} IC95%=[{ulo:.3f},{uhi:.3f}] (secundaria)")
print(f"Patologías con AP > prevalencia (=señal real): {sum(r['AP_supera_prevalencia'] for r in rows if r['label'] in PATHOLOGY_LABELS)}/{len(PATHOLOGY_LABELS)}")
pd.DataFrame(rows).to_csv(OUTPUT_DIR/"metrics_best_model.csv",index=False)

# ── B1 · métricas clínicas en los tres puntos ────────────────────────────────────────────────
clin=pd.concat([clinical_report(test_pred,y_test,m_test,PUNTOS[k],punto=nm)
                for k,nm in [("f1","f1"),("cribado","cribado_Se>=0.90"),("confirm","confirmacion_Sp>=0.90")]],ignore_index=True)
print("\n── PUNTOS DE OPERACIÓN (test) ──")
for punto in clin["punto"].unique():
    print(f"\n  · Punto '{punto}':")
    print(f"    {'Etiqueta':18s} {'thr':>5} {'Se':>6} {'Sp':>6} {'VPP':>6} {'VPN':>6} {'FN':>4} {'FP':>4}")
    for _,r in clin[clin["punto"]==punto].iterrows():
        print(f"    {r['etiqueta']:18s} {r['umbral']:5.2f} {r['Se']:6.3f} {r['Sp']:6.3f} {r['VPP']:6.3f} {r['VPN']:6.3f} {int(r['FN']):4d} {int(r['FP']):4d}")
clin.to_csv(OUTPUT_DIR/"puntos_operacion_test.csv",index=False)

# ── B6 · consistencia · B8 · estratificación ─────────────────────────────────────────────────
cons=consistency_no_finding(test_pred)
print(f"\n── B6 · corr(P(Sin hallazgo), max P(patología))={cons['corr_NoFinding_vs_maxPatologia']:.3f} "
      f"(debe ser NEGATIVA) · incoherentes={cons['pct_incoherentes']:.1f} %")
strat=stratified_report(df_test,test_pred,y_test,m_test)
print("\n── B8 · Rendimiento por subgrupo (n<60 ⇒ posible ruido) ──")
print(strat.to_string(index=False,float_format=lambda v:f"{v:.4f}"))
strat.to_csv(OUTPUT_DIR/"estratificacion_subgrupos.csv",index=False)

json.dump({"best":BEST,"primary_metric":"AUC-PR (macro_AP_path)","cfgs":CFGS,
           "test_macro_ap_path":M["macro_AP_path"],"test_macro_ap_ci":[mlo,mhi],
           "test_macro_auc_path":M["macro_AUC_path"],"test_macro_auc_ci":[ulo,uhi],
           "per_label":{l:M[l] for l in LABELS},"consistencia_no_finding":cons},
          open(OUTPUT_DIR/"summary_labs_v2.json","w",encoding="utf-8"),indent=2,default=str,ensure_ascii=False)
print("\nGuardados: metrics_best_model.csv · puntos_operacion_test.csv · estratificacion_subgrupos.csv · summary_labs_v2.json")

In [ ]:
# CELDA 13b · EXPORTAR OOF/val/test PARA EL STACKING (formato idéntico al de CXR/ECG)
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · save_predictions
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : vuelca a CSV, por split, las probabilidades crudas y calibradas de las 6 etiquetas
#              junto al hadm_id que las une con las demás modalidades.
# POR QUÉ    : el stacking multimodal debe LEER estas predicciones sin re-entrenar nada; el OOF de
#              train es imprescindible para entrenar el meta-modelo SIN FUGA.
# ENTRADAS   : df (split) · raw (N,6) · cal (N,6) · name ("oof_train"|"val"|"test")
# SALIDAS    : DataFrame guardado en OUTPUT_DIR/labs_pred_<name>.csv
# ORIGEN EDA : §11 · "Fusión: late fusion binary relevance, meta-LR por patología sobre
#              probabilidades CALIBRADAS. Comparar vs mejor mono-modelo".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              tras la fusión hay que RECALIBRAR y volver a comprobar el Brier: la calibración de
#              cada modalidad NO se conserva automáticamente al combinarlas.
# NOTA: los calibradores y apply_cal se definieron en la CELDA 13 (no se recalibra aquí).
# ══════════════════════════════════════════════════════════════════════════════
oof_train_labs = oof["Ensemble"] if BEST=="Ensemble" else oof[BEST]
def save_predictions(df, raw, cal, name):
    cols={"hadm_id": df["hadm_id"].to_numpy()}
    for j,l in enumerate(LABELS):
        key=l.replace(" ","_")
        cols[f"labs_{key}"]=raw[:,j]; cols[f"labs_{key}_cal"]=cal[:,j]
    out=pd.DataFrame(cols); path=OUTPUT_DIR/f"labs_pred_{name}.csv"; out.to_csv(path,index=False)
    print(f"   guardado {path.name}  ({out.shape[0]} filas, {out.shape[1]} cols)"); return out

save_predictions(df_train, oof_train_labs, apply_cal(oof_train_labs), "oof_train")
save_predictions(df_val,   val_pred_raw,   val_pred,                  "val")
save_predictions(df_test,  test_pred_raw,  test_pred,                 "test")
print(f"OOF/val/test del LABS v2 exportados (modelo={BEST}) → los carga directamente el stacking.")

In [ ]:
# CELDA 14 · MATRICES DE CONFUSIÓN DEL MEJOR (test, por etiqueta)
fig,axes=plt.subplots(2,3,figsize=(13,8)); fig.suptitle(f"Mejor ({BEST}) — Matrices de confusión (test)",fontweight="bold")
for j,l in enumerate(LABELS):
    ax=axes[j//3,j%3]; s=m_test[:,j]==1; yt=y_test[s,j]; yp=(test_pred[s,j]>=thr_val.get(l,0.5)).astype(int)
    if len(yt)==0: ax.axis("off"); continue
    cm=confusion_matrix(yt,yp,labels=[0,1])
    sns.heatmap(cm,annot=True,fmt="d",cmap="Purples",cbar=False,ax=ax,xticklabels=["Pred 0","Pred 1"],yticklabels=["Real 0","Real 1"])
    ax.set_title(f"{l} (thr={thr_val.get(l,0.5):.2f})",fontsize=10)
plt.tight_layout(); plt.savefig(FIG_DIR/"confusion_mejor.png",dpi=150,bbox_inches="tight"); plt.show()
print("Guardadas.")


In [ ]:
# CELDA 15 · IMPORTANCIA: Gini vs MI vs ANOVA (B5) + dependencia de los flags MNAR (B7)
# ORIGEN EDA · §10: "en Opacidad, fiarse más de MI/ANOVA que de RF-Gini por el desbalanceo";
#                   "los tres métodos coinciden → reduce el riesgo de artefacto de un método concreto".
#              §6: "vigilar que el modelo no dependa en exceso del patrón de ausencia (flags)".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#   · ¿coinciden Gini y MI/ANOVA en el top? Donde coincidan, la conclusión es sólida.
#   · ¿aparecen arriba los ratios clínicos (RDW×Edad, BUN/Creatinina)? Validaría el eje cardio-renal.
#   · ¿aparecen arriba los flags de GASOMETRÍA (pco2, po2, lactato, exceso de base)? Sería el atajo
#     asistencial "le hicieron gasometría ⇒ está crítico", NO biología. Hay que advertirlo.

# ── Importancia Gini del XGBoost tuneado, promediada sobre las 6 etiquetas ─────────────────────
imp=np.zeros(TREE_tr.shape[1])
for j in range(N_LABELS):
    sel=m_train[:,j]==1; yj=y_train[sel,j]
    if len(np.unique(yj))<2: continue
    spw=(yj==0).sum()/max((yj==1).sum(),1); clf=make_xgb(spw,CFGS.get("XGBoost",{})); clf.fit(TREE_tr[sel],yj)
    fi=clf.feature_importances_; imp+=fi/(fi.sum()+1e-9)
imp/=N_LABELS

# ── B5 · Importancia por Información Mutua y ANOVA (criterios no sesgados por el Gini) ─────────
imp_alt=importancia_mi_anova(TREE_tr,y_train,m_train,TREE_NAMES)
imp_alt.to_csv(OUTPUT_DIR/"importancia_mi_anova.csv",index=False)
print("── B5 · Top-10 por Información Mutua + ANOVA ──")
print(imp_alt.head(10).to_string(index=False))

# ── B7 · ¿cuánta importancia recae en los flags de missingness? ───────────────────────────────
dep=flag_importance_share(imp,TREE_NAMES)
print(f"\n── B7 · Importancia acumulada en flags MNAR (Gini): {dep['pct_importancia_flags']:.1f} % "
      f"(flag más influyente: {dep['flag_top']})")
if dep["pct_importancia_flags"]>25:
    print("   [AVISO] >25 %: parte del rendimiento puede ser ARTEFACTO del proceso asistencial")
    print("           ('le pidieron analítica ⇒ está grave') y no generalizaría a otro hospital.")
json.dump(dep,open(OUTPUT_DIR/"dependencia_flags_mnar.json","w",encoding="utf-8"),indent=2,ensure_ascii=False)

# ── Figuras: Gini (flags en rojo) y MI+ANOVA ──────────────────────────────────────────────────
order=np.argsort(imp)[::-1][:22]
fig,axes=plt.subplots(1,2,figsize=(16,7))
cols=["#c0392b" if str(TREE_NAMES[i]).startswith("falta:") else "#5b8db8" for i in order][::-1]
axes[0].barh([TREE_NAMES[i] for i in order][::-1],[imp[i] for i in order][::-1],color=cols)
axes[0].set_title("Importancia Gini — XGBoost (top 22)\nrojo = flag de missingness"); axes[0].tick_params(labelsize=8)
top_alt=imp_alt.head(22).iloc[::-1]
axes[1].barh(top_alt["variable"],top_alt["media"],color="#117a65"); axes[1].tick_params(labelsize=8)
axes[1].set_title("Importancia media MI + ANOVA (top 22)")
plt.tight_layout(); plt.savefig(FIG_DIR/"importancia_variables.png",dpi=150,bbox_inches="tight"); plt.show()

# ── Figura: AP obtenida frente a su línea base (la prevalencia) ───────────────────────────────
fig2,ax=plt.subplots(figsize=(9,4.5))
aps=[M[l]["AP"] for l in LABELS]; prevs=[M[l]["prevalencia"] for l in LABELS]; x=np.arange(N_LABELS)
ax.bar(x-0.2,aps,0.4,label="AUC-PR obtenida",color="#117a65")
ax.bar(x+0.2,prevs,0.4,label="prevalencia (línea base)",color="#bdc3c7")
ax.set_xticks(x); ax.set_xticklabels([l[:11] for l in LABELS],rotation=30,ha="right"); ax.set_ylim(0,1)
ax.set_title("LABS v2 · AUC-PR vs línea base (verde por encima de gris = señal real)"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/"ap_vs_prevalencia.png",dpi=150,bbox_inches="tight"); plt.show()

In [ ]:
# CELDA 16 · EQUIDAD POR SUBGRUPO (test) — sexo, etnia y tipo de ingreso
# ══ equidad por subgrupo ═════════════════════════════════════════════════════
# QUÉ HACE: recalcula la métrica primaria (AUC-PR macro) dentro de cada subgrupo demográfico.
# FINALIDAD: detectar si el modelo rinde peor en algún grupo (equidad algorítmica).
# ORIGEN EDA: §2 y §9 · recuadros naranjas "diferencias demográficas pequeñas pero reales → evaluar
#            equidad por sexo y etnia en la fase de resultados" y "monitorizar equidad" (etnia).
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): OJO al tamaño muestral — varios subgrupos del test
#            (464 pacientes) tienen n<60, así que las diferencias pueden ser RUIDO y no sesgo real.
#            No afirmar inequidad sin IC; reportar n junto a cada métrica.
dft=df_test.reset_index(drop=True); eq_rows=[]
for col,lab in [("gender","GÉNERO"),("race","ETNIA"),("admission_type","INGRESO")]:
    if col not in dft.columns: continue
    print(f"\n-- AUC-PR macro (patologías) por {lab} --")
    for v in sorted(dft[col].dropna().unique(),key=str):
        idx=dft.index[dft[col]==v].to_numpy()
        if len(idx)<15:
            print(f"   {str(v):22s} n={len(idx):4d} (insuficiente)")
            eq_rows.append({"variable":lab,"grupo":str(v),"n":len(idx),"macro_AP":np.nan,"macro_AUC":np.nan}); continue
        mm=multilabel_metrics(test_pred[idx],y_test[idx],m_test[idx])
        print(f"   {str(v):22s} n={len(idx):4d}  macroAP={mm['macro_AP_path']:.4f}  (macroAUC={mm['macro_AUC_path']:.4f})")
        eq_rows.append({"variable":lab,"grupo":str(v),"n":len(idx),"macro_AP":mm["macro_AP_path"],"macro_AUC":mm["macro_AUC_path"]})
pd.DataFrame(eq_rows).to_csv(OUTPUT_DIR/"equidad_subgrupos.csv",index=False)
print("\nEquidad completada → equidad_subgrupos.csv (recordar: n pequeños ⇒ interpretar con cautela).")

---
## ✅ Conclusión — Módulo tabular v2

Ensemble de 4 arquitecturas tuneadas con Optuna **optimizando AUC-PR**, K=5 sin leakage, ratios clínicos
(BUN/Creatinina, Neutrófilos/Linfocitos, RDW×Edad) y scores de severidad por cluster fisiológico.

Adopta el **etiquetado FINAL** (negativo = 0 + NaN→0; −1 enmascarado; sin derivar de *No Finding*),
**RobustScaler** en la rama densa, y el **kit clínico completo B1–B8**: tres puntos de operación con
Se/Sp/VPP/VPN, verificación de la calibración (Brier + fiabilidad), pre-registro de la regla de decisión,
IC bootstrap de AP y AUC, importancia MI/ANOVA, chequeo de consistencia y estratificación por subgrupos.

### 📌 Para la documentación posterior (recuadros naranjas a redactar con los resultados)
- **¿Cambia el ganador** al seleccionar por AUC-PR en vez de por AUC-ROC? Si cambia, explicarlo.
- **IC solapados**: si el ensemble y el mejor individual tienen IC que se solapan, **no** afirmar que
  el ensemble es superior. Es el techo de la señal tabular que anticipaba el EDA.
- **Puntos de operación**: contrastar los falsos negativos del punto de *cribado* con los falsos
  positivos del de *confirmación*. Si ninguno de los dos es clínicamente asumible, la conclusión es que
  el tabular **no sirve en solitario** y solo aporta en la fusión (§5/§8 del EDA).
- **Calibración**: ¿mejora el Brier tras la isotónica? Recordar que hay que **recalibrar tras la fusión**.
- **Importancia**: ¿coinciden Gini y MI/ANOVA? Vigilar si los **flags de gasometría** (pco2, po2,
  lactato, exceso de base) suben al top — sería el atajo asistencial, no biología.
- **Equidad**: diferencias por etnia con n<60 pueden ser ruido; no concluir sin IC.
- **`cxr_view`**: excluida como predictor por decisión de diseño (proxy de gravedad); se reporta solo
  estratificando. Comentar si el rendimiento difiere mucho entre AP y PA.
- Las cifras **no son comparables** con las de la ejecución anterior (otro etiquetado y otra métrica).